<a href="https://colab.research.google.com/github/narendrapatel6321-dotcom/sec-10k-crag/blob/main/notebooks/10k_pinecone_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Install Dependencies

In [1]:
!pip install -q edgartools sec-edgar-downloader langchain langchain-community langchain-chroma langchain-huggingface sentence-transformers rank_bm25 pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 118.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 

##Mount Drive & Setup Paths

In [2]:
from pathlib import Path
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/sec-10k-rag')
DATA_DIR = PROJECT_ROOT / 'data'
XBRL_DIR = DATA_DIR / 'xbrl_financials'
PROSE_DIR = DATA_DIR / 'prose_sections'
INDEX_DIR = DATA_DIR / 'index'

for directory in [DATA_DIR, XBRL_DIR, PROSE_DIR, INDEX_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# SEC identity registration
COMPANY_NAME = "Narendra Kumar"
EMAIL = "narendrapatel6321@gmail.com"

TICKERS = ["C", "AAPL", "GOOGL", "GS", "MSFT"]
FILINGS_PER_TICKER = 3

print(f"Working directory: {PROJECT_ROOT}")

Mounted at /content/drive
Working directory: /content/drive/MyDrive/sec-10k-rag


##Raw Filing Download

In [3]:
from sec_edgar_downloader import Downloader

dl = Downloader(COMPANY_NAME, EMAIL, str(DATA_DIR))

print(">>> Checking / Downloading raw 10-K filings...")
for ticker in TICKERS:
    ticker_10k_dir = DATA_DIR / "sec-edgar-filings" / ticker / "10-K"

    # Check if filings are already cached
    if ticker_10k_dir.exists():
        existing = [d for d in ticker_10k_dir.iterdir() if d.is_dir()]
        if len(existing) >= FILINGS_PER_TICKER:
            print(f"  [CACHE] {ticker}: {len(existing)} filings present. Skipping download.")
            continue

    print(f"  [DOWNLOAD] Fetching {FILINGS_PER_TICKER} filings for {ticker}...")
    try:
        dl.get("10-K", ticker, limit=FILINGS_PER_TICKER, download_details=True)
        print(f"  [SUCCESS] {ticker} downloaded.")
    except Exception as e:
        print(f"  [ERROR] {ticker}: {e}")

>>> Checking / Downloading raw 10-K filings...
  [DOWNLOAD] Fetching 3 filings for C...
  [SUCCESS] C downloaded.
  [DOWNLOAD] Fetching 3 filings for AAPL...
  [SUCCESS] AAPL downloaded.
  [DOWNLOAD] Fetching 3 filings for GOOGL...
  [SUCCESS] GOOGL downloaded.
  [DOWNLOAD] Fetching 3 filings for GS...
  [SUCCESS] GS downloaded.
  [DOWNLOAD] Fetching 3 filings for MSFT...
  [SUCCESS] MSFT downloaded.


##Structured XBRL Financials Extraction

In [4]:
from edgar import Company, set_identity
from datetime import datetime
import pandas as pd
import pickle

set_identity(f"{COMPANY_NAME} {EMAIL}")

xbrl_records = {}
xbrl_failures = []

print(">>> Extracting structured XBRL statements...")

for ticker in TICKERS:
    company = Company(ticker)
    filings = company.get_filings(form="10-K", amendments=False).head(FILINGS_PER_TICKER)

    for filing in filings:
        acc_no = filing.accession_no
        key = f"{ticker}_{acc_no}"
        try:
            xbrl = filing.xbrl()
            fin = xbrl.statements

            income_stmt = fin.income_statement().to_dataframe()
            balance_sheet = fin.balance_sheet().to_dataframe()
            cash_flow = fin.cash_flow_statement().to_dataframe()

            por = getattr(filing, "period_of_report", None)
            if por:
                if isinstance(por, str):
                    por = datetime.strptime(por, "%Y-%m-%d")
                fiscal_year = por.year
            else:
                fiscal_year = filing.filing_date.year if filing.filing_date.month > 6 else filing.filing_date.year - 1

            xbrl_records[key] = {
                "ticker": ticker,
                "fiscal_year": fiscal_year,
                "accession": acc_no,
                "income_statement": income_stmt,
                "balance_sheet": balance_sheet,
                "cash_flow": cash_flow,
            }

            out_dir = XBRL_DIR / key
            out_dir.mkdir(exist_ok=True)
            income_stmt.to_csv(out_dir / "income_statement.csv", index=False)
            balance_sheet.to_csv(out_dir / "balance_sheet.csv", index=False)
            cash_flow.to_csv(out_dir / "cash_flow.csv", index=False)

            print(f"  [XBRL] {key} — FY{fiscal_year}")
        except Exception as e:
            print(f"  [ERROR] {key}: {e}")
            xbrl_failures.append((ticker, acc_no, str(e)))

with open(XBRL_DIR / "all_xbrl_records.pkl", "wb") as f:
    pickle.dump(xbrl_records, f)

print(f"\nTotal XBRL records saved: {len(xbrl_records)}")

>>> Extracting structured XBRL statements...
  [XBRL] C_0000831001-26-000011 — FY2025
  [XBRL] C_0000831001-25-000067 — FY2024
  [XBRL] C_0000831001-24-000033 — FY2023
  [XBRL] AAPL_0000320193-25-000079 — FY2025
  [XBRL] AAPL_0000320193-24-000123 — FY2024
  [XBRL] AAPL_0000320193-23-000106 — FY2023
  [XBRL] GOOGL_0001652044-26-000018 — FY2025
  [XBRL] GOOGL_0001652044-25-000014 — FY2024
  [XBRL] GOOGL_0001652044-24-000022 — FY2023
  [XBRL] GS_0000886982-26-000091 — FY2025
  [XBRL] GS_0000886982-25-000005 — FY2024
  [XBRL] GS_0000886982-24-000006 — FY2023
  [XBRL] MSFT_0001193125-26-323660 — FY2026
  [XBRL] MSFT_0000950170-25-100235 — FY2025
  [XBRL] MSFT_0000950170-24-087843 — FY2024

Total XBRL records saved: 15


##Prose Extraction (Item 1A & Item 7)

In [5]:
prose_records = {}
prose_failures = []

print(">>> Extracting MD&A and Risk Factors...")

for ticker in TICKERS:
    company = Company(ticker)
    filings = company.get_filings(form="10-K", amendments=False).head(FILINGS_PER_TICKER)

    for filing in filings:
        acc_no = filing.accession_no
        key = f"{ticker}_{acc_no}"
        try:
            tenk = filing.obj()
            risk_factors = tenk["Item 1A"]
            mdna = tenk["Item 7"]

            prose_records[key] = {
                "ticker": ticker,
                "accession": acc_no,
                "risk_factors": risk_factors,
                "mdna": mdna,
            }

            print(f"  [PROSE] {key}: Risk Factors={len(risk_factors) if risk_factors else 0} chars, MD&A={len(mdna) if mdna else 0} chars")

            if not risk_factors:
                prose_failures.append((ticker, acc_no, "risk_factors_missing"))
            if not mdna:
                prose_failures.append((ticker, acc_no, "mdna_missing"))

        except Exception as e:
            print(f"  [ERROR] {key}: {e}")
            prose_failures.append((ticker, acc_no, str(e)))

with open(PROSE_DIR / "all_prose_records.pkl", "wb") as f:
    pickle.dump(prose_records, f)

print(f"\nTotal Prose records saved: {len(prose_records)}")
print(f"Extraction issues: {prose_failures}")

>>> Extracting MD&A and Risk Factors...
  [PROSE] C_0000831001-26-000011: Risk Factors=88529 chars, MD&A=407746 chars
  [PROSE] C_0000831001-25-000067: Risk Factors=96990 chars, MD&A=446022 chars
  [PROSE] C_0000831001-24-000033: Risk Factors=100297 chars, MD&A=433098 chars
  [PROSE] AAPL_0000320193-25-000079: Risk Factors=68163 chars, MD&A=18018 chars
  [PROSE] AAPL_0000320193-24-000123: Risk Factors=68887 chars, MD&A=15358 chars
  [PROSE] AAPL_0000320193-23-000106: Risk Factors=67998 chars, MD&A=15509 chars
  [PROSE] GOOGL_0001652044-26-000018: Risk Factors=85311 chars, MD&A=52545 chars
  [PROSE] GOOGL_0001652044-25-000014: Risk Factors=83432 chars, MD&A=58046 chars
  [PROSE] GOOGL_0001652044-24-000022: Risk Factors=76171 chars, MD&A=59669 chars
  [PROSE] GS_0000886982-26-000091: Risk Factors=142138 chars, MD&A=310379 chars
  [PROSE] GS_0000886982-25-000005: Risk Factors=144556 chars, MD&A=294910 chars
  [PROSE] GS_0000886982-24-000006: Risk Factors=141927 chars, MD&A=291378 chars
  

##Chunking & Hybrid Index Creation (ChromaDB + BM25)

In [7]:
!pip install -q langchain-text-splitters

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
import pickle

print(">>> Loading prose sections for chunking...")
with open(PROSE_DIR / "all_prose_records.pkl", "rb") as f:
    prose_records = pickle.load(f)

# 1. Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

docs = []
for key, record in prose_records.items():
    for section_name in ["risk_factors", "mdna"]:
        content = record[section_name]
        if content:
            chunks = text_splitter.split_text(content)
            for i, chunk in enumerate(chunks):
                docs.append(Document(
                    page_content=chunk,
                    metadata={
                        "ticker": record["ticker"],
                        "accession": record["accession"],
                        "section": section_name,
                        "chunk_id": i
                    }
                ))

print(f"Total chunks created: {len(docs)}")

# 2. Dense Vector Store (bge-small-en-v1.5)
print("Generating dense embeddings and building ChromaDB...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=str(INDEX_DIR / "chroma_db")
)
print("ChromaDB saved to disk.")

# 3. Sparse Index (BM25)
print("Building and saving BM25 retriever...")
bm25_retriever = BM25Retriever.from_documents(docs)
with open(INDEX_DIR / "bm25_retriever.pkl", "wb") as f:
    pickle.dump(bm25_retriever, f)

print("BM25 index saved to disk.")
print("\n>>> Ingestion & Indexing Pipeline Complete!")

/tmp/ipykernel_1654/2475450099.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


>>> Loading prose sections for chunking...
Total chunks created: 5613
Generating dense embeddings and building ChromaDB...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB saved to disk.
Building and saving BM25 retriever...
BM25 index saved to disk.

>>> Ingestion & Indexing Pipeline Complete!
